# 04 P&L and budget

Lessons 8-9: company, kitchen and item P&L for Sep 2025 to Aug 2026, actual vs budget, and what to do per item and kitchen.

Data: `orders_enriched.parquet` (notebook 02) + `waste_daily` with the filled rows from `load_tables()`. Main GP = GP-C. Overhead is never split to items.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets the notebook import src/

from src.load_clean import load_tables
from src.pnl import (SUMS, add_ratios, as_statement, budget_variance, company_pnl, contribution,
                     gp_bridge, kitchen_pnl, margin_gap, menu_matrix)
from src.pnl_excel import FormulaCheck, write_pnl_xlsx

# Same chart setup as notebooks 02-03.
pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["fb"] = go.layout.Template(layout=dict(margin=dict(t=118, l=70, r=30, b=60), font_size=13))
pio.templates.default = "plotly_white+fb"
OUT = ROOT / "outputs"

ITEMS = ["Watermelon", "Pineapple", "Guava", "PassionFruit", "MixedBerryPremium"]
ITEM_COLORS = dict(zip(ITEMS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
UP, DOWN, TOTAL, MID = "#2a78d6", "#e34948", "#52514e", "#f0efec"
MENU_MONTHS = [f"2026-{i:02d}" for i in range(3, 9)]  # all 5 items on sale


def subtitle(title, sub):
    """Chart title: the finding (split at ';' to fit), then a grey line saying what is plotted."""
    sub_html = f"<span style='font-size:12px;color:#52514e'>{sub}</span>"
    return dict(text=title.replace("; ", ";<br>") + "<br>" + sub_html, font_size=16,
                x=0.01, xanchor="left", y=1, yref="container", yanchor="top", pad=dict(t=36))


def thb_k(df, cols=None, digits=0):
    """THB columns in thousands, for display only."""
    out = df.copy()
    for col in cols or out.select_dtypes("number").columns:
        out[col] = (out[col] / 1000).round(digits)
    return out


sheets = load_tables()
e = pd.read_parquet(ROOT / "data" / "processed" / "orders_enriched.parquet")
c = contribution(e, sheets)        # month x kitchen x sku
k = kitchen_pnl(c, sheets)         # month x kitchen, with overhead
m = company_pnl(k)                 # month
fy = m[SUMS + ["overhead", "operating_profit"]].sum()
print(f"{len(c):,} month-kitchen-item rows | {len(k)} kitchen-months | {len(m)} months")

216 month-kitchen-item rows | 48 kitchen-months | 12 months


## 1. Does the P&L tie out?

In [2]:
w = sheets["waste_daily"]
oh = sheets["kitchen_overhead"]
checks = pd.DataFrame([
    ("Gross revenue", fy["gross_revenue"], 31_146_932.57, "notebook 01 clean total"),
    ("CM (GP-B)", fy["cm"], e["cm_thb"].sum(), "enriched cm_thb"),
    ("Waste cost", fy["waste_cost"], w["est_waste_cost_thb"].sum(), "waste_daily, filled"),
    ("Overhead", fy["overhead"], oh["fixed_monthly_overhead_thb"].sum() * 12, "4 kitchens x 12 months"),
    ("Cups", fy["cups"], e["units_sold"].sum(), "enriched units_sold"),
], columns=["line", "pnl_thb", "source_thb", "source"])
checks["diff"] = checks["pnl_thb"] - checks["source_thb"]
display(checks.round(2))

assert np.isclose(fy["gross_revenue"], 31_146_932.57, atol=0.01)
assert np.isclose(fy["cm"], e["cm_thb"].sum())
assert np.isclose(fy["waste_cost"], w["est_waste_cost_thb"].sum())
assert abs(fy["waste_cost"] - 860_300) < 1_000
assert oh["fixed_monthly_overhead_thb"].sum() == 670_000 and np.isclose(fy["overhead"], 670_000 * 12)
# Sum of kitchens = company, and sum of items = company, every month and every line.
assert np.allclose(k.groupby("month")[SUMS].sum(), m.set_index("month")[SUMS])
assert np.allclose(c.groupby("month")[SUMS].sum(), m.set_index("month")[SUMS])
assert np.allclose(m["gross_profit"], m["cm"] - m["waste_cost"])
assert np.allclose(m["operating_profit"], m["gross_profit"] - m["overhead"])

filled = w.loc[w["is_filled"], "est_waste_cost_thb"].sum()
waste_only = c[(c["cups"] == 0) & (c["waste_units"] > 0)]
print(f"All ties pass. Filled waste: {filled:,.0f} THB. Waste rows with no sales that month: {len(waste_only)}")

,line,pnl_thb,source_thb,source,diff
0,Gross revenue,31146932.57,31146932.57,notebook 01 clean total,0.0
1,CM (GP-B),8496619.65,8496619.65,enriched cm_thb,0.0
2,Waste cost,860307.21,860307.21,"waste_daily, filled",0.0
3,Overhead,8040000.00,8040000.00,4 kitchens x 12 months,0.0
4,Cups,618123.00,618123.00,enriched units_sold,0.0


All ties pass. Filled waste: 4,459 THB. Waste rows with no sales that month: 0


**Revenue 31.15M, CM 8.50M, waste 860.3k (4.5k of it filled) and overhead 8.04M all match their sources to 0.01 THB.** Kitchens and items both add up to the company total in every month.

## 2. Which GP definition fits the budget?

In [3]:
budget = sheets["monthly_budget"]
budget_pct = budget["budget_gross_profit_thb"].sum() / budget["budget_revenue_thb"].sum() * 100
defs = pd.DataFrame({
    "deducts": ["fruit, pack+labor", "+ commission", "+ waste"],
    "thb": [fy["gp_a"], fy["gp_b"], fy["gross_profit"]],
}, index=["GP-A", "GP-B (CM)", "GP-C"])
defs["pct_of_revenue"] = defs["thb"] / fy["gross_revenue"] * 100
defs["pts_vs_budget"] = defs["pct_of_revenue"] - budget_pct
print(f"Budget GP %: {budget_pct:.1f}")
display(thb_k(defs, ["thb"]).round(1))

Budget GP %: 25.9


,deducts,thb,pct_of_revenue,pts_vs_budget
GP-A,"fruit, pack+labor",17014.0,54.6,28.7
GP-B (CM),+ commission,8497.0,27.3,1.4
GP-C,+ waste,7636.0,24.5,-1.4


**GP-A 54.6% is far from the 25.9% budget, so the budget must be after commission. GP-B 27.3% and GP-C 24.5% sit either side of it.**
Main = GP-C: waste is a real variable cost of selling, and the budget note does not say it is left out. GP-B is shown as a check.

## 3. What does the company P&L look like?

In [4]:
statement = as_statement(m)
thb_lines = statement.index[~statement.index.str.contains("%")]
show = statement.copy()
show.loc[thb_lines] = (show.loc[thb_lines] / 1000).round(0)
show.loc[~show.index.isin(thb_lines)] = show.loc[~show.index.isin(thb_lines)].round(1)
print("THB thousands; % lines in %")
display(show)
m.to_csv(OUT / "company_pnl.csv", index=False)
k.to_csv(OUT / "kitchen_pnl.csv", index=False)
c.to_csv(OUT / "item_kitchen_pnl.csv", index=False)

THB thousands; % lines in %


month,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,FY
Cups sold,40.0,41.0,46.0,50.0,50.0,48.0,67.0,77.0,67.0,45.0,43.0,44.0,618.0
List revenue,1968.0,2012.0,2285.0,2459.0,2488.0,2368.0,3631.0,4092.0,3539.0,2367.0,2254.0,2270.0,31732.0
Discount,55.0,45.0,30.0,34.0,29.0,43.0,48.0,62.0,60.0,65.0,57.0,58.0,586.0
Gross revenue,1913.0,1967.0,2255.0,2425.0,2458.0,2325.0,3583.0,4029.0,3479.0,2303.0,2197.0,2212.0,31147.0
Commission,519.0,538.0,615.0,665.0,674.0,637.0,979.0,1110.0,950.0,631.0,600.0,601.0,8517.0
Net revenue,1394.0,1429.0,1640.0,1760.0,1784.0,1688.0,2605.0,2919.0,2529.0,1672.0,1597.0,1611.0,22630.0
Fruit cost,409.0,413.0,466.0,506.0,504.0,483.0,855.0,886.0,769.0,603.0,588.0,608.0,7089.0
Packaging + labor,446.0,456.0,517.0,557.0,563.0,536.0,782.0,893.0,776.0,519.0,496.0,501.0,7044.0
Contribution margin (GP-B),539.0,560.0,657.0,697.0,717.0,669.0,968.0,1140.0,985.0,549.0,513.0,502.0,8497.0
Waste cost,35.0,36.0,41.0,43.0,44.0,43.0,137.0,133.0,111.0,84.0,76.0,78.0,860.0


In [5]:
n_loss = (m["operating_profit"] < 0).sum()
fig = go.Figure(go.Bar(
    x=m["month"], y=m["operating_profit"] / 1000,
    marker_color=np.where(m["operating_profit"] >= 0, UP, DOWN),
    text=(m["operating_profit"] / 1000).round(0).astype(int), textposition="outside",
    hovertemplate="%{x}<br>%{y:,.0f}k THB<extra></extra>"))
fig.update_layout(
    title=subtitle(f"Operating profit is below 0 in {n_loss} of 12 months; only the Mar-May peak covers the 670k overhead",
                   f"Operating profit (GP-C - kitchen overhead), THB thousands. FY {fy['operating_profit'] / 1000:,.0f}k"),
    yaxis_title="THB thousands", xaxis_title=None, showlegend=False, height=460)
fig.show()

**FY operating profit -404k THB (-1.3% of revenue).** GP-C 7.64M does not cover 8.04M overhead.
8 months below 0. The budget itself (8.4M GP) leaves only 0.36M after overhead, so even hitting it is near break-even.
Commission is the biggest cost line (27% of revenue), bigger than fruit (23%).

## 4. Which months miss the budget, and why?

In [6]:
v = budget_variance(m, sheets)
assert np.allclose(v["revenue_effect"] + v["margin_effect"], v["gp_var"])
cols = ["gross_revenue", "budget_revenue", "revenue_var", "revenue_var_pct", "gross_profit", "budget_gp",
        "gp_var", "gp_var_pct", "gp_pct", "revenue_effect", "margin_effect"]
display(thb_k(v.set_index("month")[cols],
              ["gross_revenue", "budget_revenue", "revenue_var", "gross_profit", "budget_gp", "gp_var",
               "revenue_effect", "margin_effect"]).round(1))
tot = v[["gross_revenue", "budget_revenue", "gross_profit", "budget_gp", "gp_var", "revenue_effect",
         "margin_effect"]].sum()
assert np.isclose(tot["revenue_effect"] + tot["margin_effect"], tot["gp_var"])
print(f"FY: revenue {tot['gross_revenue'] / 1e6:.2f}M vs {tot['budget_revenue'] / 1e6:.1f}M | "
      f"GP {tot['gross_profit'] / 1e6:.2f}M vs {tot['budget_gp'] / 1e6:.1f}M | "
      f"variance {tot['gp_var'] / 1000:,.0f}k = revenue effect {tot['revenue_effect'] / 1000:,.0f}k "
      f"+ margin effect {tot['margin_effect'] / 1000:,.0f}k")
v.to_csv(OUT / "budget_variance.csv", index=False)

,gross_revenue,budget_revenue,revenue_var,revenue_var_pct,gross_profit,budget_gp,gp_var,gp_var_pct,gp_pct,revenue_effect,margin_effect
month,,,,,,,,,,,
2025-09,1913.0,2700.0,-787.0,-29.1,505.0,700.0,-195.0,-27.9,26.4,-204.0,9.0
2025-10,1967.0,2700.0,-733.0,-27.1,525.0,700.0,-175.0,-25.0,26.7,-190.0,15.0
2025-11,2255.0,2700.0,-445.0,-16.5,616.0,700.0,-84.0,-12.0,27.3,-115.0,32.0
2025-12,2425.0,2700.0,-275.0,-10.2,654.0,700.0,-46.0,-6.6,27.0,-71.0,25.0
2026-01,2458.0,2700.0,-242.0,-9.0,673.0,700.0,-27.0,-3.9,27.4,-63.0,36.0
2026-02,2325.0,2700.0,-375.0,-13.9,626.0,700.0,-74.0,-10.5,26.9,-97.0,23.0
2026-03,3583.0,2700.0,883.0,32.7,831.0,700.0,131.0,18.7,23.2,229.0,-98.0
2026-04,4029.0,2700.0,1329.0,49.2,1008.0,700.0,308.0,44.0,25.0,345.0,-37.0
2026-05,3479.0,2700.0,779.0,28.9,873.0,700.0,173.0,24.8,25.1,202.0,-29.0


FY: revenue 31.15M vs 32.4M | GP 7.64M vs 8.4M | variance -764k = revenue effect -325k + margin effect -439k


In [7]:
miss = v.loc[v["gp_var"] < 0, "month"]
fig = go.Figure()
fig.add_bar(x=v["month"], y=v["gross_profit"] / 1000, name="Actual GP-C", marker_color=UP,
            hovertemplate="%{x}<br>Actual %{y:,.0f}k<extra></extra>")
fig.add_scatter(x=v["month"], y=v["budget_gp"] / 1000, name="Budget GP", mode="lines",
                line=dict(color=TOTAL, width=2, dash="dash"), hovertemplate="Budget %{y:,.0f}k<extra></extra>")
worst = v.nsmallest(3, "gp_var")
fig.update_layout(
    title=subtitle(f"GP misses budget in {len(miss)} of 12 months; Jun-Aug miss most "
                   f"({-worst['gp_var'].max() / 1000:,.0f}k to {-worst['gp_var'].min() / 1000:,.0f}k a month)",
                   "Gross profit (GP-C) vs flat budget of 700k a month, THB thousands"),
    yaxis_title="THB thousands", height=460, legend=dict(orientation="h", y=1.02, x=1, xanchor="right"))
fig.show()

**GP is below budget in 9 of 12 months, -764k for the year (-9.1%).** Revenue effect -325k, margin effect -439k.
Sep-Feb: revenue 9-29% under a flat budget (low season), but margin is above 25.9% (26-27%), so GP misses less than revenue.
Mar-May: revenue +29-49% (berry launch, Songkran) beats budget even with a lower margin.
Jun-Aug: both miss. Revenue -15 to -19% and margin 19-20%. Why: see the cost lines below.

In [8]:
lines = m.set_index("month")[["gross_revenue", "commission", "fruit", "pack_labor", "waste_cost"]]
cost_pct = (lines.drop(columns="gross_revenue").div(lines["gross_revenue"], axis=0) * 100).round(1)
cost_pct["gp_pct"] = m.set_index("month")["gp_pct"].round(1)
# Old-item fruit cost per cup, so the berry mix does not hide the season.
old = e[e["sku"] != "MixedBerryPremium"].assign(month=lambda d: d["date"].dt.strftime("%Y-%m"))
fruit_cup = old.groupby("month").apply(lambda d: d["fruit_cost_thb"].sum() / d["units_sold"].sum(),
                                       include_groups=False)
cost_pct["old_items_fruit_per_cup"] = fruit_cup.round(2)
display(cost_pct)

,commission,fruit,pack_labor,waste_cost,gp_pct,old_items_fruit_per_cup
month,,,,,,
2025-09,27.1,21.4,23.3,1.8,26.4,10.29
2025-10,27.3,21.0,23.2,1.8,26.7,10.18
2025-11,27.3,20.7,22.9,1.8,27.3,10.12
2025-12,27.4,20.9,23.0,1.8,27.0,10.20
2026-01,27.4,20.5,22.9,1.8,27.4,10.05
2026-02,27.4,20.8,23.1,1.8,26.9,10.11
2026-03,27.3,23.9,21.8,3.8,23.2,9.92
2026-04,27.5,22.0,22.2,3.3,25.0,9.30
2026-05,27.3,22.1,22.3,3.2,25.1,9.24


**Jun-Aug fruit is 26-28% of revenue vs 21% in Sep-Feb.** Two causes: old-item fruit cost 11.1-11.6 THB a cup (rainy season) vs 9.2-9.3 in Apr-May, and berry (fruit about 38% of its price) in the mix.
Waste 1.8% of revenue before berry, 3.2-3.8% after. Commission (27%) and pack+labor stay flat. The flat budget ignores season and fruit cost (its own note says so).

## 5. Which items drag the margin?

In [9]:
ig = margin_gap(c, v, "sku")
ig.to_csv(OUT / "item_gap.csv", index=False)
item = c.groupby("sku")[["cups", "gross_revenue", "commission", "fruit", "pack_labor", "waste_cost",
                         "gross_profit"]].sum().reindex(ITEMS)
item["gp_pct"] = item["gross_profit"] / item["gross_revenue"] * 100
item["gap_vs_budget"] = ig.groupby("sku")["gap"].sum()
assert np.isclose(item["gap_vs_budget"].sum(), v["margin_effect"].sum())

# Why: each cost line's share of revenue vs the company mix, in THB. The lines add up exactly to
# the item's gap vs the company GP %.
co = fy[["commission", "fruit", "pack_labor", "waste_cost"]] / fy["gross_revenue"]
excess = -(item[co.index].div(item["gross_revenue"], axis=0) - co).mul(item["gross_revenue"], axis=0)
gap_vs_company = item["gross_profit"] - item["gross_revenue"] * fy["gross_profit"] / fy["gross_revenue"]
assert np.allclose(excess.sum(axis=1), gap_vs_company)


def cause_text(sku):
    """Cost lines more than 100k THB above the company mix, for items below the budget margin."""
    if item.loc[sku, "gap_vs_budget"] >= 0:
        return "none: above budget margin"
    worst = excess.loc[sku][excess.loc[sku] < -100_000].sort_values()
    return ", ".join(f"{line} {val / 1000:,.0f}k vs mix" for line, val in worst.items())


item["main_cause"] = [cause_text(s) for s in item.index]
show = thb_k(item, ["gross_revenue", "gross_profit", "gap_vs_budget"])
display(show.join(thb_k(excess).add_prefix("vs_mix_")).round(1))

,cups,gross_revenue,commission,fruit,pack_labor,waste_cost,gross_profit,gp_pct,gap_vs_budget,main_cause,vs_mix_commission,vs_mix_fruit,vs_mix_pack_labor,vs_mix_waste_cost
sku,,,,,,,,,,,,,,
Watermelon,244991,10813.0,2955404.7,1888115.3,2694901.0,113746.0,3161.0,29.2,357.0,none: above budget margin,1.0,573.0,-250.0,185.0
Pineapple,162168,7816.0,2135749.8,1522883.0,1783848.0,114442.8,2259.0,28.9,233.0,none: above budget margin,2.0,256.0,-16.0,101.0
Guava,101504,4883.0,1338075.1,944937.5,1167296.0,89602.5,1343.0,27.5,77.0,none: above budget margin,-3.0,166.0,-63.0,45.0
PassionFruit,81416,5196.0,1424252.7,1659807.8,976992.0,207579.0,928.0,17.9,-420.0,fruit -477k vs mix,-3.0,-477.0,198.0,-64.0
MixedBerryPremium,28044,2439.0,663909.3,1073480.8,420660.0,334936.8,-54.0,-2.2,-686.0,"fruit -518k vs mix, waste_cost -268k vs mix",3.0,-518.0,131.0,-268.0


In [10]:
bridge = gp_bridge(v, ig)
bridge.to_csv(OUT / "gp_bridge.csv", index=False)
assert np.isclose(bridge.loc[bridge["kind"] == "step", "thb"].sum() + bridge["thb"].iloc[0], bridge["thb"].iloc[-1])
print(thb_k(bridge, ["thb"], 1).to_string(index=False))

measure = np.where(bridge["kind"] == "total", "total", "relative")
measure[0] = "absolute"
fig = go.Figure(go.Waterfall(
    x=bridge["step"].str.replace(" margin gap", ""), y=bridge["thb"] / 1000, measure=measure,
    text=(bridge["thb"] / 1000).round(0).astype(int), textposition="outside",
    increasing=dict(marker_color=UP), decreasing=dict(marker_color=DOWN), totals=dict(marker_color=TOTAL),
    connector=dict(line=dict(color="#c3c2b7", width=1)),
    hovertemplate="%{x}<br>%{y:,.0f}k THB<extra></extra>"))
gaps = bridge.set_index("step")["thb"]
fig.update_layout(
    title=subtitle(f"Berry ({gaps['MixedBerryPremium margin gap'] / 1000:,.0f}k) and PassionFruit "
                   f"({gaps['PassionFruit margin gap'] / 1000:,.0f}k) cost more GP than the revenue shortfall "
                   f"({gaps['Revenue effect'] / 1000:,.0f}k)",
                   "Budget GP to actual GP-C, full year, THB thousands (axis starts at 6,800). Item bar = item GP - item revenue x 25.9%"),
    yaxis=dict(title="THB thousands", range=[6800, 8900]), showlegend=False, height=500)
fig.show()

                        step  kind    thb
                   Budget GP total 8400.0
              Revenue effect  step -324.9
MixedBerryPremium margin gap  step -686.2
     PassionFruit margin gap  step -419.6
            Guava margin gap  step   77.2
        Pineapple margin gap  step  232.6
       Watermelon margin gap  step  357.2
                   Actual GP total 7636.3


**Berry -686k and PassionFruit -420k: the two items below the 25.9% budget margin cost 1.1M GP, more than the -325k revenue shortfall.**
Berry: GP-C -2.2% of revenue in 6 months; fruit and waste are the lines above the company mix. PassionFruit 17.9%: fruit cost.
Watermelon, Pineapple and Guava earn 27-29% and add +667k above budget margin.

## 6. Which kitchens make money after overhead?

In [11]:
kfy = add_ratios(k.groupby("kitchen")[SUMS + ["overhead", "operating_profit"]].sum())
kfy["revenue_per_month"] = kfy["gross_revenue"] / 12
kfy["op_per_month"] = kfy["operating_profit"] / 12
kfy["months_below_0"] = k.groupby("kitchen")["operating_profit"].apply(lambda s: (s < 0).sum())
# Revenue a month that covers overhead at the kitchen's own GP %.
kfy["breakeven_revenue_per_month"] = kfy["overhead"] / 12 / (kfy["gp_pct"] / 100)
kg = margin_gap(c, v, "kitchen")
kfy["gap_vs_budget"] = kg.groupby("kitchen")["gap"].sum()
assert np.isclose(kfy["gap_vs_budget"].sum(), v["margin_effect"].sum())
assert np.isclose(kfy["operating_profit"].sum(), fy["operating_profit"])
cols = ["revenue_per_month", "breakeven_revenue_per_month", "gp_pct", "op_per_month", "op_margin_pct",
        "months_below_0", "waste_rate_pct", "gap_vs_budget"]
display(thb_k(kfy[cols], ["revenue_per_month", "breakeven_revenue_per_month", "op_per_month",
                          "gap_vs_budget"]).round(1).sort_values("op_per_month"))

,revenue_per_month,breakeven_revenue_per_month,gp_pct,op_per_month,op_margin_pct,months_below_0,waste_rate_pct,gap_vs_budget
kitchen,,,,,,,,
BKK_Ladprao,384.0,609.0,23.0,-52.0,-13.5,12,8.3,-135.0
BKK_Sukhumvit,642.0,805.0,24.8,-41.0,-6.3,9,5.1,-84.0
Pattaya_Beach,720.0,608.0,24.7,28.0,3.8,5,5.1,-107.0
Pattaya_Central,850.0,725.0,24.8,31.0,3.6,5,5.1,-113.0


In [12]:
heat = k.pivot(index="kitchen", columns="month", values="operating_profit").loc[kfy["op_per_month"].sort_values().index]
lim = np.abs(heat.to_numpy()).max() / 1000
fig = go.Figure(go.Heatmap(
    z=heat / 1000, x=heat.columns, y=heat.index, zmid=0, zmin=-lim, zmax=lim,
    colorscale=[[0, DOWN], [0.5, MID], [1, UP]], xgap=2, ygap=2,
    text=(heat / 1000).round(0).astype(int), texttemplate="%{text}",
    colorbar=dict(title="THB k"), hovertemplate="%{y} %{x}<br>%{z:,.0f}k THB<extra></extra>"))
below = kfy["months_below_0"]
fig.update_layout(
    title=subtitle(f"BKK_Ladprao loses money in all {below['BKK_Ladprao']} months and BKK_Sukhumvit in "
                   f"{below['BKK_Sukhumvit']}; Pattaya kitchens lose only in the rainy months",
                   "Operating profit per kitchen and month (GP-C - fixed overhead), THB thousands"),
    height=420, yaxis_title=None)
fig.show()

In [13]:
# Is waste set by demand or by habit? Old items only; berry has its own row in section 8.
oldk = c[c["sku"] != "MixedBerryPremium"].groupby("kitchen")[["cups", "waste_units", "waste_cost"]].sum()
oldk["cups_sold_per_day"] = oldk["cups"] / 365
oldk["cups_wasted_per_day"] = oldk["waste_units"] / 365
oldk["waste_rate_pct"] = oldk["waste_units"] / (oldk["cups"] + oldk["waste_units"]) * 100
display(oldk[["cups_sold_per_day", "cups_wasted_per_day", "waste_rate_pct"]].round(1))

# Ladprao excess waste if it wasted at the rate of the other 3 kitchens, item by item.
ck = c[c["sku"] != "MixedBerryPremium"].groupby(["kitchen", "sku"])[["cups", "waste_units", "waste_cost"]].sum()
lad = ck.loc["BKK_Ladprao"]
others = ck.drop(index="BKK_Ladprao").groupby("sku").sum()
rate = others["waste_units"] / (others["cups"] + others["waste_units"])
target_units = lad["cups"] * rate / (1 - rate)
lad_saving = ((lad["waste_units"] - target_units) * lad["waste_cost"] / lad["waste_units"]).sum() / 12
print(f"Ladprao old-item waste at the other kitchens' rate: saves {lad_saving / 1000:.1f}k THB a month")

,cups_sold_per_day,cups_wasted_per_day,waste_rate_pct
kitchen,,,
BKK_Ladprao,239.1,17.8,6.9
BKK_Sukhumvit,400.0,18.2,4.3
Pattaya_Beach,448.4,20.4,4.4
Pattaya_Central,529.2,24.3,4.4


Ladprao old-item waste at the other kitchens' rate: saves 3.7k THB a month


**Pattaya kitchens make +28k to +31k a month after overhead; BKK_Sukhumvit -41k and BKK_Ladprao -52k.** Company total -34k a month.
GP % is about the same in all 4 (23-25%); the problem is volume vs overhead. Ladprao needs 609k revenue a month to break even and sells 384k; Sukhumvit needs 806k and sells 642k.
All 4 kitchens are below the budget margin (-84k to -135k a year), because berry and PassionFruit sell everywhere.
Ladprao throws away 18 old-item cups a day, the same as Sukhumvit, which sells 1.7x more (6.9% vs 4.3% waste rate): prep looks fixed, not set by demand. At the other kitchens' rate it saves 3.7k a month; that does not close a 52k gap.
Overhead note says "central kitchen" on all 4 rows; looks like a copy. Amounts differ, so used as given.

## 7. Menu engineering: which items are Stars and which are Dogs? (Mar-Aug 2026)

In [14]:
mm = menu_matrix(c, MENU_MONTHS).set_index("sku").reindex(ITEMS)
mmk = menu_matrix(c, MENU_MONTHS, by=["kitchen"])
assert mm["cups"].sum() == mmk["cups"].sum()
display(mm[["cups_per_day", "gp_per_cup", "waste_rate_pct", "menu_class"]].round(2))
print(f"Cut lines: {mm['pop_line'].iloc[0]:.0f} cups a day (mean of items), "
      f"{mm['profit_line'].iloc[0]:.2f} THB GP-C per cup (cup-weighted)")
display(mmk.pivot(index="kitchen", columns="sku", values="menu_class")[ITEMS])
pd.concat([mm.reset_index().assign(kitchen="All"), mmk]).to_csv(OUT / "menu_matrix.csv", index=False)

,cups_per_day,gp_per_cup,waste_rate_pct,menu_class
sku,,,,
Watermelon,715.65,13.14,3.24,Star
Pineapple,472.04,14.42,4.31,Star
Guava,294.23,12.83,5.44,Puzzle
PassionFruit,236.01,9.55,8.81,Dog
MixedBerryPremium,152.41,-1.92,20.54,Dog


Cut lines: 374 cups a day (mean of items), 11.73 THB GP-C per cup (cup-weighted)


sku,Watermelon,Pineapple,Guava,PassionFruit,MixedBerryPremium
kitchen,,,,,
BKK_Ladprao,Star,Star,Puzzle,Dog,Dog
BKK_Sukhumvit,Star,Star,Puzzle,Dog,Dog
Pattaya_Beach,Star,Star,Puzzle,Dog,Dog
Pattaya_Central,Star,Star,Puzzle,Dog,Dog


In [15]:
fig = go.Figure()
for sku, r in mm.iterrows():
    fig.add_scatter(x=[r["cups_per_day"]], y=[r["gp_per_cup"]], mode="markers+text", name=sku,
                    marker=dict(size=14, color=ITEM_COLORS[sku], line=dict(color="#fcfcfb", width=2)),
                    text=[f"{sku} ({r['menu_class']})"], textposition="top center",
                    hovertemplate=f"{sku}<br>%{{x:.0f}} cups a day<br>%{{y:.2f}} THB GP per cup<extra></extra>")
pop, prof = mm["pop_line"].iloc[0], mm["profit_line"].iloc[0]
fig.add_vline(x=pop, line=dict(color=TOTAL, dash="dash", width=1))
fig.add_hline(y=prof, line=dict(color=TOTAL, dash="dash", width=1))
fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
fig.update_layout(
    title=subtitle("Watermelon and Pineapple are Stars; berry is the only item that loses money per cup after waste",
                   "Mar-Aug 2026. Cups sold per day vs GP-C per cup after waste. Dashed = averages (cut lines)"),
    xaxis=dict(title="Cups sold per day", range=[0, 800]), yaxis=dict(title="GP-C per cup, THB", range=[-4, 17]),
    showlegend=False, height=500)
fig.show()

**Stars: Watermelon (716 cups a day, 13.1 THB a cup) and Pineapple (472, 14.4). Puzzle: Guava (294, 12.8). Dogs: PassionFruit (236, 9.6) and berry (152, -1.9).**
Same classes in all 4 kitchens. Berry is negative in every kitchen, worst in Ladprao (-9.5 THB a cup, 30% waste).
PassionFruit is a Dog by the lines but still +9.6 THB a cup, so the fix is price (lesson 6), not delisting.

## 8. What do we do per item and per kitchen?

In [16]:
price = pd.read_csv(OUT / "price_recommendation.csv").set_index("sku")
promo = pd.read_csv(OUT / "promo_decision.csv")
rc101 = promo[promo["code"] == "RC101"].set_index("sku")
other_promo = promo[promo["code"] != "RC101"]


def promo_parts(sku):
    """Lesson 7 RC101 value for one item: (kept for sure, extra if the test passes)."""
    r = rc101.loc[sku]
    return r["kill_saving"], (0.0 if pd.isna(r["modify_net"]) else r["modify_net"])


rows = []
for sku in ITEMS:
    p = price.loc[sku]
    kill, modify = promo_parts(sku)
    gp_cup, cls, wr = mm.loc[sku, "gp_per_cup"], mm.loc[sku, "menu_class"], mm.loc[sku, "waste_rate_pct"]
    cause = item.loc[sku, "main_cause"]
    problem = f"{cls}: GP-C {gp_cup:.1f} THB a cup, {item.loc[sku, 'gp_pct']:.1f}% vs budget 25.9%"
    if p["decision"] == "raise" and cause.startswith("none"):
        problem += f"; demand allows a higher price (elasticity {p['elasticity']:.2f}, break-even {p['breakeven_elasticity']:.2f})"
    elif p["decision"] == "keep + test":
        problem += f"; price room unsure (CI {p['ci_low']:.2f} to {p['ci_high']:.2f}, break-even {p['breakeven_elasticity']:.2f})"
    if sku == "MixedBerryPremium":
        fix = p["delist_thb"] + p["cm_after_both_low"]
        low, high = fix, p["delist_thb"]
        action = (f"Raise 89 -> 99 THB and cut waste {wr:.0f}% -> 10% now; delist if GP per cup is still "
                  f"below 0 after 8 weeks. Stop the RC101 berry bundle (inside these numbers)")
        note = "low = fix (still about -4.6k a month after it), high = delist; never add both"
    elif p["decision"] == "raise":
        low, high = p["cm_change_low"] + kill, p["cm_change_high"] + kill
        action = f"Raise {p['price_now']:.0f} -> {p['new_price']:.0f} THB; RC101: {rc101.loc[sku, 'decision']}"
        note = "price before waste on the Mar-Aug base (April peak inside); promo at the new price"
    else:
        low, high = kill, p["cm_change_if_test_high"] + modify
        action = (f"Keep {p['price_now']:.0f}, 4-week test at {p['test_price']:.0f} THB in one kitchen; "
                  f"RC101: {rc101.loc[sku, 'decision']} ({rc101.loc[sku, 'change']})")
        note = "low = tests fail and we go back (0); high = both tests pass"
    rows.append(dict(level="item", name=sku, problem=problem, cause=cause, action=action, thb_month_low=low, thb_month_high=high, in_plan=True,
                     source="lesson 6 price + lesson 7 RC101", confidence=p["confidence"], note=note))

rows.append(dict(level="company", name="Promos RC102 + RC103",
                 problem=f"lose {-other_promo['net_per_month_new_prices'].sum() / 1000:.1f}k THB CM a month",
                 cause="discount", action="Kill RC103 on both platforms; RC102 test 12% without berry",
                 thb_month_low=other_promo["kill_saving"].sum(),
                 thb_month_high=other_promo["kill_saving"].sum() + other_promo["modify_net"].fillna(0).sum(),
                 in_plan=True, source="lesson 7", confidence="high (kill) / low (test)",
                 note="berry part overlaps the berry row"))

# Kitchen rows: their share of the item plan (by Mar-Aug cups), not added again.
share = mmk.pivot(index="kitchen", columns="sku", values="cups")
share = share / share.sum()
item_low = pd.Series({r["name"]: r["thb_month_low"] for r in rows if r["level"] == "item"})
item_high = pd.Series({r["name"]: r["thb_month_high"] for r in rows if r["level"] == "item"})
cup_share = share.mul(mm["cups"]).sum(axis=1) / mm["cups"].sum()
promo_low_k = rows[-1]["thb_month_low"] * cup_share
for kname, r in kfy.sort_values("op_per_month").iterrows():
    n_below = int(r["months_below_0"])
    k_low = (share.loc[kname] * item_low).sum() + promo_low_k[kname]
    k_high = (share.loc[kname] * item_high).sum() + rows[-1]["thb_month_high"] * cup_share[kname]
    after = r["op_per_month"] + k_low
    if r["op_per_month"] < 0:
        problem = (f"op profit {r['op_per_month'] / 1000:.1f}k a month, below 0 in {n_below} months; "
                   f"sells {r['revenue_per_month'] / 1000:.0f}k vs {r['breakeven_revenue_per_month'] / 1000:.0f}k needed")
        cause = "overhead vs volume"
    else:
        problem = f"op profit +{r['op_per_month'] / 1000:.1f}k a month; below 0 in {n_below} rainy months"
        cause = "fruit cost in rainy months"
    action = "Item plan above" + ("; review overhead (rent, staff hours) if still below 0 after Q4"
                                  if after < 0 else "")
    rows.append(dict(level="kitchen", name=kname, problem=problem, cause=cause, action=action,
                     thb_month_low=k_low, thb_month_high=k_high, in_plan=False,
                     source="share of item rows", confidence="as item rows",
                     note=f"after the low plan: {after / 1000:+.1f}k a month; inside the item rows"))

rows.append(dict(level="kitchen", name="BKK_Ladprao waste",
                 problem="wastes as many old-item cups a day (18) as kitchens that sell 1.7-2.2x more",
                 cause="waste_cost", action="Prep from demand (lesson 10 rules)",
                 thb_month_low=0.0, thb_month_high=lad_saving, in_plan=False, source="this notebook",
                 confidence="medium", note="lesson 10 sizes prep rules for all kitchens; use that number, not both"))
rows.append(dict(level="kitchen", name="BKK_Ladprao close (review only)",
                 problem="below 0 in all 12 months", cause="overhead vs volume",
                 action="Not now. Review if Q4 is still below 0",
                 thb_month_low=0.0, thb_month_high=-kfy.loc["BKK_Ladprao", "op_per_month"], in_plan=False,
                 source="this notebook", confidence="low",
                 note="high = all overhead can be cut and no customer moves to another kitchen"))

portfolio = pd.DataFrame(rows)
portfolio[["thb_month_low", "thb_month_high"]] = portfolio[["thb_month_low", "thb_month_high"]].round(-2)
portfolio.to_csv(OUT / "portfolio_decision.csv", index=False)
pd.set_option("display.max_colwidth", 90)
display(thb_k(portfolio.set_index(["level", "name"]), ["thb_month_low", "thb_month_high"], 1)
        [["problem", "cause", "action", "thb_month_low", "thb_month_high", "in_plan", "confidence"]])

plan = portfolio[portfolio["in_plan"]]
kitchen_rows = portfolio[(portfolio["level"] == "kitchen") & portfolio["name"].isin(kfy.index)]
assert np.isclose(kitchen_rows["thb_month_low"].sum(), plan["thb_month_low"].sum(), atol=500)
gap = -v["gp_var"].sum() / 12
print(f"Plan: +{plan['thb_month_low'].sum() / 1000:.1f}k (low) to +{plan['thb_month_high'].sum() / 1000:.1f}k (high) "
      f"THB a month | GP gap to budget {gap / 1000:.1f}k a month | operating loss "
      f"{-fy['operating_profit'] / 12 / 1000:.1f}k a month")

problem  \
level   name                                                                                                                         
item    Watermelon                       Star: GP-C 13.1 THB a cup, 29.2% vs budget 25.9%; demand allows a higher price (elasti...   
        Pineapple                        Star: GP-C 14.4 THB a cup, 28.9% vs budget 25.9%; price room unsure (CI -2.12 to -0.51...   
        Guava                            Puzzle: GP-C 12.8 THB a cup, 27.5% vs budget 25.9%; price room unsure (CI -2.60 to -0....   
        PassionFruit                                                                Dog: GP-C 9.6 THB a cup, 17.9% vs budget 25.9%   
        MixedBerryPremium                                                          Dog: GP-C -1.9 THB a cup, -2.2% vs budget 25.9%   
company Promos RC102 + RC103                                                                              lose 9.6k THB CM a month   
kitchen BKK_Ladprao                                      op profit -51.7k a month, below 0 in 12 months; sells 384k vs 609k needed   
        BKK_Sukhumvit                                     op profit -40.5k a month, below 0 in 9 months; sells 642k vs 805k needed   
        Pattaya_Beach                                                          op profit +27.7k a month; below 0 in 5 rainy months   
        Pattaya_Central                                                        op profit +31.0k a month; below 0 in 5 rainy months   
        BKK_Ladprao waste                              wastes as many old-item cups a day (18) as kitchens that sell 1.7-2.2x more   
        BKK_Ladprao close (review only)                                                                   below 0 in all 12 months   

                                                                               cause  \
level   name                                                                           
item    Watermelon                                         none: above budget margin   
        Pineapple                                          none: above budget margin   
        Guava                                              none: above budget margin   
        PassionFruit                                              fruit -477k vs mix   
        MixedBerryPremium                fruit -518k vs mix, waste_cost -268k vs mix   
company Promos RC102 + RC103                                                discount   
kitchen BKK_Ladprao                                               overhead vs volume   
        BKK_Sukhumvit                                             overhead vs volume   
        Pattaya_Beach                                     fruit cost in rainy months   
        Pattaya_Central                                   fruit cost in rainy months   
        BKK_Ladprao waste                                                 waste_cost   
        BKK_Ladprao close (review only)                           overhead vs volume   

                                                                                                                            action  \
level   name                                                                                                                         
item    Watermelon                                                                                 Raise 45 -> 49 THB; RC101: Kill   
        Pineapple                        Keep 49, 4-week test at 55 THB in one kitchen; RC101: Modify + test (quiet hours only ...   
        Guava                            Keep 49, 4-week test at 55 THB in one kitchen; RC101: Modify + test (shallower discoun...   
        PassionFruit                                                                               Raise 65 -> 69 THB; RC101: Kill   
        MixedBerryPremium                Raise 89 -> 99 THB and cut waste 21% -> 10% now; delist if GP per cup is still below 0...   
company Promos RC102 + RC103                                            Kil

Plan: +70.5k (low) to +191.4k (high) THB a month | GP gap to budget 63.6k a month | operating loss 33.6k a month


**Plan: +70.5k (low) to +191.4k (high) a month, on the Mar-Aug base.** The low end alone covers the average GP gap to budget (64k) and the operating loss (34k).
It does not cover a Jun-Aug month (GP 235-276k under budget), and the base includes the April peak, so Sep-Nov gains will be smaller: the scenario model (lesson 12) checks this.
Berry is the biggest single line (+39k fix, +43k delist). Kitchen rows are shares of the item rows: after the low plan both BKK kitchens are still below 0.

## 9. Does the Excel file match the notebook?

In [17]:
xlsx = OUT / "FruitBlend24_PnL.xlsx"
layout = write_pnl_xlsx(xlsx, m, k, c, v, oh)
fc = FormulaCheck(xlsx)
rows, col = layout["company_rows"], layout["fy_col"]
st = as_statement(m)
for key, label in [("rev", "Gross revenue"), ("net", "Net revenue"), ("cm", "Contribution margin (GP-B)"),
                   ("gp", "Gross profit (GP-C)"), ("op", "Operating profit")]:
    assert np.isclose(fc.value("Company P&L", f"{col}{rows[key]}"), st.loc[label, "FY"]), label
    for j, month in enumerate(m["month"]):
        letter = chr(ord("B") + j)
        assert np.isclose(fc.value("Company P&L", f"{letter}{rows[key]}"), st.loc[label, month]), (label, month)
for name, kr in layout["kitchen_rows"].items():
    assert np.isclose(fc.value("Kitchen P&L", f"{col}{kr['op']}"), kfy.loc[name, "operating_profit"]), name
r = layout["item_total_row"]
assert np.isclose(fc.value("Item x Kitchen", f"K{r}"), fy["gross_profit"])
assert abs(fc.value("Item x Kitchen", f"K{r + 1}")) < 0.01
for i in range(13):
    assert abs(fc.value("Budget variance", f"N{5 + i}")) < 0.01
assert np.isclose(fc.value("Budget variance", "H17"), v["gp_var"].sum())
fy_gp = fc.value("Company P&L", f"{col}{rows['gp']}")
print(f"Excel formulas match: FY GP-C {fy_gp:,.2f} | "
      f"variance checks 0 in 13 rows | sheets: {fc.wb.sheetnames}")

Excel formulas match: FY GP-C 7,636,312.43 | variance checks 0 in 13 rows | sheets: ['Company P&L', 'Kitchen P&L', 'Item x Kitchen', 'Budget variance', 'Definitions']


## Findings -> next steps
1. FY operating profit -404k THB: GP-C 7.64M (24.5%) vs overhead 8.04M; even the 8.4M budget GP leaves only 0.36M -> judge the plan on operating profit, not on hitting budget.
2. GP -764k vs budget: revenue effect -325k, margin effect -439k; Jun-Aug alone -774k (fruit 26-28% of revenue vs 21%) -> plan rainy-season fruit cost in the Sep-Nov forecast (lesson 11).
3. Berry (-686k) and PassionFruit (-420k) margin gaps are bigger than the revenue miss -> berry 99 THB + waste to 10% now, delist review after 8 weeks (+39k to +43k a month); PassionFruit 69 THB (+12k to +18k).
4. BKK_Ladprao (-52k a month, 12 of 12 months below 0) and BKK_Sukhumvit (-41k) lose money on volume, not margin; both stay below 0 after the low plan -> set Q4 targets and review overhead if missed.
5. Ladprao wastes 18 old-item cups a day, as many as Sukhumvit that sells 1.7x more (6.9% vs 4.3%) -> prep from demand (lesson 10); worth 3.7k a month, so it does not fix Ladprao alone.